# 06 - Fine-tuning TrOCR (microsoft/trocr-base-handwritten)

Zero-shot TrOCR scored 0.382 on an informal 8-row check (`notebooks/05_baseline_crnn.ipynb`'s
comparison section, see README Progress) - far below the CRNN's 0.702, failing by hallucinating
fluent modern-English sentences rather than transcribing literally (its RoBERTa-initialized
decoder falling back on its language-model prior). This notebook tests whether fine-tuning on
our own fit split closes that gap, using the same fixed 120-row val sample and evaluation
harness as the CRNN notebooks for a direct comparison. PaddleOCR and Kraken were dropped
(see README Next steps) - only TrOCR is being pursued as the pretrained candidate.

In [2]:
import sys
sys.path.append('..')

import copy
import time

import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

from src.config import MODELS_DIR, RANDOM_STATE
from src.data import build_image_path, load_raw_data, split_train_val
from src.evaluate import combined_score

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [3]:
train = load_raw_data('train')
fit_df, val_df = split_train_val(train)
fit_df = fit_df.sample(200, random_state=RANDOM_STATE).reset_index(drop=True)  # smoke test only
val_sample = val_df.sample(20, random_state=RANDOM_STATE).reset_index(drop=True)  # smoke test only
len(fit_df), len(val_df), len(val_sample)

(200, 818, 120)

In [4]:
MODEL_NAME = 'microsoft/trocr-base-handwritten'

processor = TrOCRProcessor.from_pretrained(MODEL_NAME)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME).to(device)

# Required for generate() to work correctly during fine-tuning (HF TrOCR convention)
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.weight | MISSING | 
encoder.pooler.dense.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [5]:
MAX_TARGET_LENGTH = 128


class TrOCRDataset(Dataset):
    def __init__(self, df, processor):
        self.ids = df['ID'].tolist()
        self.targets = df['Target'].tolist()
        self.processor = processor

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        image = Image.open(build_image_path(self.ids[i])).convert('RGB')
        pixel_values = self.processor(image, return_tensors='pt').pixel_values.squeeze(0)
        labels = self.processor.tokenizer(
            self.targets[i], padding='max_length', max_length=MAX_TARGET_LENGTH, truncation=True
        ).input_ids
        # Pad-token labels must be -100 so the loss ignores them (HF convention)
        labels = [l if l != self.processor.tokenizer.pad_token_id else -100 for l in labels]
        return {'pixel_values': pixel_values, 'labels': torch.tensor(labels)}

In [6]:
BATCH_SIZE = 4  # reduce to 4 if you hit CUDA out-of-memory on the 4060's 8GB

fit_ds = TrOCRDataset(fit_df, processor)
fit_dl = DataLoader(fit_ds, batch_size=BATCH_SIZE, shuffle=True)

val_ds = TrOCRDataset(val_sample, processor)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

In [7]:
from tqdm import tqdm


def state_dict_to_cpu(model):
    """CPU copy of the model's weights - keeps the GPU free of a second,
    redundant full-precision copy of a 334M-param model throughout training."""
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def evaluate(loader, reference_texts):
    model.eval()
    hyps = []
    with torch.no_grad(), torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        for batch in tqdm(loader, desc='eval'):
            pixel_values = batch['pixel_values'].to(device)
            generated_ids = model.generate(pixel_values, max_new_tokens=MAX_TARGET_LENGTH)
            hyps.extend(processor.batch_decode(generated_ids, skip_special_tokens=True))
    model.train()
    return combined_score(reference_texts, hyps)


def train_with_early_stopping(model, optimizer, max_epochs, patience, initial_best_score=-1.0):
    t0 = time.time()
    history = []
    best_score = initial_best_score
    best_epoch = 0
    best_state = state_dict_to_cpu(model)
    epochs_since_improvement = 0

    for epoch in range(1, max_epochs + 1):
        total_loss = 0.0
        for batch in tqdm(fit_dl, desc=f'train epoch {epoch}'):
            pixel_values = batch['pixel_values'].to(device)
            labels = batch['labels'].to(device)
            optimizer.zero_grad()
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                loss = model(pixel_values=pixel_values, labels=labels).loss
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * pixel_values.size(0)
        train_loss = total_loss / len(fit_ds)

        score = evaluate(val_dl, val_sample['Target'].tolist())
        history.append({'epoch': epoch, 'train_loss': train_loss, **score})
        print(f"epoch {epoch:2d} | loss {train_loss:.3f} | val CER {score['cer_weighted']:.3f} "
              f"WER {score['wer_weighted']:.3f} final {score['score']:.3f} | {time.time()-t0:.0f}s")

        if score['score'] > best_score:
            best_score = score['score']
            best_epoch = epoch
            best_state = state_dict_to_cpu(model)
            epochs_since_improvement = 0
        else:
            epochs_since_improvement += 1
            if epochs_since_improvement >= patience:
                print(f"Early stopping at epoch {epoch} (no improvement for {patience} epochs)")
                break

    return best_state, best_score, best_epoch, history

In [8]:
RECORDED_CRNN_SCORE = 0.702
MAX_EPOCHS = 1
PATIENCE = 4
LR = 5e-5  # standard fine-tuning LR for a pretrained seq2seq transformer, ~20x below from-scratch training

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

best_state, best_score, best_epoch, history = train_with_early_stopping(
    model, optimizer, max_epochs=MAX_EPOCHS, patience=PATIENCE
)

KeyboardInterrupt: 

In [ ]:
model.load_state_dict(best_state)
final_score = evaluate(val_dl, val_sample['Target'].tolist())
print(f"best epoch: {best_epoch}")
print(f"TrOCR fine-tuned score: {final_score['score']:.3f} (CRNN reference was {RECORDED_CRNN_SCORE:.3f})")
print(f"delta: {final_score['score'] - RECORDED_CRNN_SCORE:+.3f}")

MODELS_DIR.mkdir(parents=True, exist_ok=True)
if final_score['score'] > RECORDED_CRNN_SCORE:
    save_path = MODELS_DIR / 'trocr_finetuned'
    model.save_pretrained(save_path)
    processor.save_pretrained(save_path)
    print(f"Saved new reference model: {save_path}")
else:
    print(f"No improvement over the CRNN ({RECORDED_CRNN_SCORE}) - keeping crnn_regularized.pt as reference. "
          "Log this as a negative result in README.md.")